# Milestone 4 - Deep Learning Implementation

Computer vision - image classification 

**Dataset:** [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification). The task is to classify natural scene images into six classes: buildings, forest, glacier, mountain, sea, and street.

## 1: Imports

I am using **TensorFlow/Keras** because it provides a production-friendly stack for defining models, building `tf.data` pipelines, monitoring metrics with `model.fit`, and exporting deployment artifacts.

Using:

- `tf.data` and `tf.keras.utils.image_dataset_from_directory` for scalable data loading
- Keras preprocessing layers for normalization and augmentation
- Keras Functional API for clear model architecture
- `model.fit` with validation metrics and checkpoints
- Keras `.keras`, TensorFlow SavedModel, and TensorFlow Lite export for deployment preparation

In [1]:
from pathlib import Path
import json
import os
import random
import time

os.environ.setdefault("MPLCONFIGDIR", str(Path(os.environ.get("TMPDIR", "/tmp")) / "tou_nn_matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available devices:", [device.name for device in tf.config.list_physical_devices()])

TensorFlow: 2.21.0
Available devices: ['/physical_device:CPU:0']


## 2: Robust Data Pipeline

I am using `Intel Image Classification` data already stored in the local `data/` directory. Layout of the directory is as follows:

```text
data/
  seg_train/seg_train/<class>/*.jpg
  seg_test/seg_test/<class>/*.jpg
  seg_pred/seg_pred/*.jpg
```

The pipeline below handles the nested Kaggle folder structure, image resizing, train/validation/test splits, normalization, augmentation, caching, and prefetching.

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 128
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print(f"Root: {ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Artifacts: {ARTIFACT_DIR}")

Root: /Users/harshkyada/Desktop/ToU NN Challenge
Data dir: /Users/harshkyada/Desktop/ToU NN Challenge/data
Artifacts: /Users/harshkyada/Desktop/ToU NN Challenge/artifacts


In [ ]:
def resolve_split_dir(base_dir: Path, split_name: str) -> Path:
    candidates = [
        base_dir / split_name / split_name,
        base_dir / split_name,
        base_dir,
    ]
    for candidate in candidates:
        if candidate.exists() and any(p.is_dir() for p in candidate.iterdir()):
            return candidate
    raise FileNotFoundError(f"Could not find split folder for {split_name} under {base_dir}")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Expected dataset directory not found: {DATA_DIR}")

train_root = resolve_split_dir(DATA_DIR, "seg_train")
test_root = resolve_split_dir(DATA_DIR, "seg_test")

train_ds = keras.utils.image_dataset_from_directory(
    train_root,
    validation_split=0.15,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)
val_ds = keras.utils.image_dataset_from_directory(
    train_root,
    validation_split=0.15,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)
test_ds = keras.utils.image_dataset_from_directory(
    test_root,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.04),
        layers.RandomZoom(0.08),
    ],
    name="data_augmentation",
)

@tf.autograph.experimental.do_not_convert
def augment_batch(images, labels):
    return data_augmentation(images, training=True), labels

train_ds = train_ds.cache().shuffle(1024, seed=SEED)
train_ds = train_ds.map(augment_batch, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

print("Classes:", class_to_idx)

Found 14034 files belonging to 6 classes.
Using 11929 files for training.
Found 14034 files belonging to 6 classes.
Using 2105 files for validation.
Found 3000 files belonging to 6 classes.
Classes: {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}


In [4]:
def show_batch(ds, class_names, max_images=8):
    images, labels = next(iter(ds))
    images = images[:max_images].numpy().astype("uint8")
    labels = labels[:max_images].numpy()
    fig, axes = plt.subplots(1, len(images), figsize=(14, 3))
    for ax, img, label in zip(axes, images, labels):
        ax.imshow(img)
        ax.set_title(class_names[int(label)])
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "milestone4_sample_batch.png", dpi=160)
    plt.show()

show_batch(train_ds, class_names)

/var/folders/8f/j9bb2nzs27g8wwq5kb3dhkxm0000gn/T/ipykernel_58726/4161364568.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3: Build and Train a Domain-Specific Model

The architecture is a compact CNN designed for natural-scene classification. It uses a deterministic Keras preprocessing layer, repeated convolution blocks, batch normalization, max pooling, global average pooling, dropout, and a final softmax classifier. This is smaller than a production transfer-learning model, but it is transparent and practical for capstone implementation.

In [5]:
def conv_block(x, filters: int):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return x


def build_scene_cnn(num_classes: int, image_size: int = 128) -> keras.Model:
    inputs = keras.Input(shape=(image_size, image_size, 3), name="image")
    x = layers.Rescaling(1.0 / 255.0, name="rescale")(inputs)
    x = conv_block(x, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    return keras.Model(inputs, outputs, name="SceneCNN")

model = build_scene_cnn(num_classes=len(class_names), image_size=IMG_SIZE)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=2e-3, weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
model.summary()

Model: "SceneCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescale (Rescaling)             │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 32, 32, 128)    │             

 Total params: 289,126 (1.10 MB)

 Trainable params: 288,230 (1.10 MB)

 Non-trainable params: 896 (3.50 KB)

In [6]:
EPOCHS = 5
checkpoint_path = ARTIFACT_DIR / "milestone4_scene_cnn_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(ARTIFACT_DIR / "milestone4_training_history.csv"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", range(1, len(history_df) + 1))
history_df

Epoch 1/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4935 - loss: 1.2624
Epoch 1: val_accuracy improved from None to 0.50784, saving model to /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_best.keras

Epoch 1: finished saving model to /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_best.keras
373/373 ━━━━━━━━━━━━━━━━━━━━ 690s 2s/step - accuracy: 0.5593 - loss: 1.0939 - val_accuracy: 0.5078 - val_loss: 1.2930
Epoch 2/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6608 - loss: 0.8728
Epoch 2: val_accuracy improved from 0.50784 to 0.51924, saving model to /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_best.keras

Epoch 2: finished saving model to /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_best.keras
373/373 ━━━━━━━━━━━━━━━━━━━━ 486s 1s/step - accuracy: 0.6783 - loss: 0.8429 - val_accuracy: 0.5192 - val_loss: 1.6858
Epoch 3/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/

,epoch,accuracy,loss,val_accuracy,val_loss
0,1,0.559309,1.093934,0.507838,1.292971
1,2,0.678263,0.842927,0.519240,1.685788
2,3,0.739039,0.703837,0.648931,0.938698
3,4,0.774331,0.628969,0.740143,0.703931
4,5,0.792355,0.579369,0.701188,0.751698


In [7]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["accuracy"], marker="o", label="train accuracy")
plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("SceneCNN Training Progress")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "milestone4_training_curve.png", dpi=160)
plt.show()

/var/folders/8f/j9bb2nzs27g8wwq5kb3dhkxm0000gn/T/ipykernel_58726/3967498822.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
best_model = keras.models.load_model(checkpoint_path)

test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
print({"test_loss": float(test_loss), "test_accuracy": float(test_accuracy)})

y_true = []
y_pred = []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

target_names = [idx_to_class[i] for i in range(len(idx_to_class))]
report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(target_names)), target_names, rotation=45, ha="right")
plt.yticks(range(len(target_names)), target_names)
plt.colorbar(label="count")
plt.title("Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "milestone4_confusion_matrix.png", dpi=160)
plt.show()

with open(ARTIFACT_DIR / "milestone4_metrics.json", "w") as f:
    json.dump(
        {
            "class_to_idx": class_to_idx,
            "test_loss": float(test_loss),
            "test_accuracy": float(test_accuracy),
            "classification_report": report,
        },
        f,
        indent=2,
    )

{'test_loss': 0.7457818388938904, 'test_accuracy': 0.7243333458900452}
              precision    recall  f1-score   support

   buildings       0.66      0.82      0.73       437
      forest       0.98      0.81      0.89       474
     glacier       0.58      0.86      0.69       553
    mountain       0.76      0.64      0.70       525
         sea       0.93      0.42      0.58       510
      street       0.71      0.79      0.75       501

    accuracy                           0.72      3000
   macro avg       0.77      0.73      0.72      3000
weighted avg       0.77      0.72      0.72      3000



/var/folders/8f/j9bb2nzs27g8wwq5kb3dhkxm0000gn/T/ipykernel_58726/4246901400.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4: Deployment

A basic deployment workflow needs a stable inference artifact and a repeatable preprocessing contract. There are three TensorFlow-friendly artifact exports:

- `.keras` model file for Keras reloads
- TensorFlow SavedModel for serving-oriented workflows
- TensorFlow Lite model for lightweight/mobile/edge deployment

In [9]:
keras_path = ARTIFACT_DIR / "milestone4_scene_cnn.keras"
saved_model_dir = ARTIFACT_DIR / "milestone4_scene_cnn_savedmodel"
tflite_path = ARTIFACT_DIR / "milestone4_scene_cnn.tflite"
labels_path = ARTIFACT_DIR / "milestone4_labels.json"

best_model.save(keras_path)
best_model.export(saved_model_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
tflite_path.write_bytes(tflite_model)
labels_path.write_text(json.dumps(idx_to_class, indent=2))

print("Keras model:", keras_path)
print("SavedModel:", saved_model_dir)
print("TFLite model:", tflite_path)
print("Labels:", labels_path)

INFO:tensorflow:Assets written to: /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_savedmodel/assets


INFO:tensorflow:Assets written to: /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_savedmodel/assets


Saved artifact at '/Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  4924640528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924644752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924644560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924644944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924640144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924640720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924648976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924648784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924644368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4924649168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  492

W0000 00:00:1779869737.452904 13082329 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1779869737.453113 13082329 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1779869737.456770 13082329 reader.cc:83] Reading SavedModel from: /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_savedmodel
I0000 00:00:1779869737.458263 13082329 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1779869737.458268 13082329 reader.cc:147] Reading SavedModel debug info (if present) from: /Users/harshkyada/Desktop/ToU NN Challenge/artifacts/milestone4_scene_cnn_savedmodel
I0000 00:00:1779869737.468533 13082329 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1779869737.470356 13082329 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1779869737.531913 13082329 loader.cc:220] Running initialization op on SavedModel bundle at path: /Users/harshkyada/Desktop/ToU NN Challenge/ar

In [10]:
def predict_image(image_path: Path, model_path: Path = checkpoint_path):
    inference_model = keras.models.load_model(model_path)
    img = keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras.utils.img_to_array(img)
    batch = np.expand_dims(arr, axis=0)
    probs = inference_model.predict(batch, verbose=0)[0]
    top_idx = int(np.argmax(probs))
    return {"label": idx_to_class[top_idx], "confidence": float(probs[top_idx])}

sample_file = next(test_root.rglob("*.jpg"))
print("Sample:", sample_file)
predict_image(sample_file)

Sample: /Users/harshkyada/Desktop/ToU NN Challenge/data/seg_test/seg_test/forest/23933.jpg


{'label': 'forest', 'confidence': 0.9979220032691956}

### Basic FastAPI Deployment Sketch

Minimal API for deployment. It accepts an uploaded image, applies the same resizing contract, and returns the predicted scene class.

```python
from fastapi import FastAPI, UploadFile
from tensorflow import keras
from PIL import Image
import numpy as np
import io

app = FastAPI()
model = keras.models.load_model("artifacts/milestone4_scene_cnn.keras")
labels = {0: "buildings", 1: "forest", 2: "glacier", 3: "mountain", 4: "sea", 5: "street"}

@app.post('/predict')
async def predict(file: UploadFile):
    image = Image.open(io.BytesIO(await file.read())).convert("RGB").resize((128, 128))
    batch = np.expand_dims(np.asarray(image, dtype=np.float32), axis=0)
    probs = model.predict(batch, verbose=0)[0]
    top_idx = int(np.argmax(probs))
    return {'label': labels[top_idx], 'confidence': float(probs[top_idx])}
```


## 5: Real-World Constraints

**Compute and latency:** A compact CNN reduces memory and inference latency compared with large pretrained models. On limited hardware, use smaller images, batch inference, TensorFlow Lite quantization, pruning, or distillation. For higher accuracy, compare against a transfer-learning baseline such as MobileNetV3 or EfficientNet-B0.

**Data quality:** Real image data often has inconsistent sizes, corrupt files, duplicates, and distribution shift. We are using TensorFlow's image-folder loader, consistent resizing, Keras normalization, caching, prefetching, and explicit class tracking. In production, we should add duplicate detection and continuous monitoring for class distribution drift.

**Fairness and bias:** Scene datasets may overrepresent specific geographies, weather, lighting, building styles, and camera qualities. A model trained on this dataset may perform worse on underrepresented regions or low-light/mobile images. Mitigations include auditing per-class and per-region performance, collecting more representative data, and documenting known failure modes.

**Privacy:** Images may contain people, addresses, vehicle plates, or location clues. Production systems should minimize retention, blur sensitive content when possible, restrict access to raw images, and define clear data deletion policies.

**Energy usage:** Training deep models consumes energy, especially when repeated frequently or run on large accelerators. This design uses a small architecture and modest image resolution to reduce compute. During production we should track training time, reuse pretrained weights when appropriate, run experiments efficiently, and avoid unnecessary retraining.

**Deployment compatibility:** Keras is best for Python/TensorFlow workflows, SavedModel is useful for serving, and TensorFlow Lite is better for mobile and edge devices. Before release, comparing exported-model predictions with the original Keras model and benchmark latency/memory on target hardware.